# 05 - Model inference
**Stage:** show every (image, variant) to a vision-language model and store the raw answers.

| | |
|---|---|
| **Input** | `data/dataset.json` |
| **Output** | `results/<model>/raw_outputs.jsonl` (one line per answer; safe to stop and re-run - finished answers are skipped) |
| **Code used** | `src/models.py`, `src/prompts.py` |

**Needs a GPU** (Colab: Runtime -> Change runtime type -> T4 GPU). `PROVIDER = "qwen"` is the main model; the second model
(LLaVA) is run in notebook 09 and the extra small-dot variants in notebook 08. `"mock"` gives FAKE answers - only to test the pipeline.

In [ ]:
# Setup: find the repo root so `from src import ...` works (run this notebook from inside the repo).
import sys
from pathlib import Path
ROOT = next((p for p in [Path.cwd(), Path.cwd().parent] if (p / "src").is_dir()), None)
assert ROOT is not None, "Run from inside the repo (the folder containing src/). In Colab: upload the repo zip, unzip it and %cd into it."
sys.path.insert(0, str(ROOT))
from src import config, utils
print("repo root:", ROOT)

In [ ]:
# Colab only
!pip -q install -U transformers accelerate bitsandbytes pillow tqdm

In [ ]:
PROVIDER = "qwen"            # "qwen" | "llava" | "gemini" | "mock" (FAKE answers, never report them)
VARIANTS = config.VARIANTS   # the 8 main variants; use config.ALL_VARIANTS to include the Phase 3 small-dot variants
LIMIT_IMAGES = None          # e.g. 2 for a quick test, None = all images
SECONDS_BETWEEN_CALLS = 0    # ~6 for the Gemini free tier

from src import models
DATA = utils.load_json(config.DATASET_JSON)
models.run_experiment(DATA, PROVIDER, VARIANTS, limit_images=LIMIT_IMAGES, sleep=SECONDS_BETWEEN_CALLS)

In [ ]:
# Peek at a few raw answers
import json
path = config.results_dir(PROVIDER) / "raw_outputs.jsonl"
for line in path.read_text().splitlines()[:4]:
    r = json.loads(line)
    print(r["image"], r["variant"], r["status"], "|", (r["raw_text"] or r["error"] or "")[:200].replace("\n", " "))

**Next:** `06_evaluation.ipynb`